# pyskills.files
> Functions for searching, creating, and reading files.

In [ ]:
#| default_exp files

In [ ]:
#| export
import difflib,re
from pathlib import Path
from tempfile import TemporaryDirectory
from fnmatch import translate

from fastcore.utils import globtastic
from fastcore.meta import splice_sig, delegates

In [ ]:
from fastcore.test import test_eq

In [ ]:
#| export
def file_view(
    path:str, # Path to view (expands `~` if needed)
    startline:int=1, # Starting line to view
    endline:int=None # End line (defaults to last line if None)
):
    "Read file contents, optionally limited to 1-based line range"
    path = Path(path).expanduser()
    lines = path.read_text().splitlines()
    if not lines: return ''
    if endline is None: endline = len(lines)
    if endline < 0: endline = len(lines)+endline+1
    startline = min(max(1, startline), len(lines))
    endline = max(startline, min(endline, len(lines)))
    return '\n'.join(f'{i}: {l}' for i,l in enumerate(lines[startline-1:endline], startline))

In [ ]:
_tmp = TemporaryDirectory()
_test_path = f'{_tmp.name}/test.txt'
_test_content = 'alpha\nbeta\ngamma\ndelta\n'
def _test_txt(): return Path(_test_path).read_text()
Path(_test_path).write_text(_test_content)
_test_path

'/var/folders/29/d2vp2sq93sb8j3k3fnkw805r0000gn/T/tmptk3mpiex/test.txt'

In [ ]:
print(file_view(_test_path, 2, 3))

2: beta
3: gamma


In [ ]:
#| export
def file_create(
    path:str, # Path to create (expands `~` if needed)
    contents:str # Contents of file to create
):
    "Create a new file with contents. Error if file exists."
    path = Path(path).expanduser()
    if path.exists(): return f'error: File exists: {path}'
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(contents)
    diff = '\n'.join(list(difflib.unified_diff([], contents.splitlines(), n=1, lineterm=''))[2:])
    return diff or 'none: Empty file created.'

In [ ]:
_new = Path(_tmp.name)/'demo.txt'
print(file_create(_new, 'one\ntwo\nthree\n'))

@@ -0,0 +1,3 @@
+one
+two
+three


In [ ]:
#| export
def find_files(
    path:Path|str='.', # path to start searching
    file_glob:str=None, # Only include files matching glob
    file_re:str=None, # Only include files matching regex
    folder_re:str=None, # Only enter folders matching regex
    skip_file_glob:str=None, # Skip files matching glob
    skip_file_re:str=None, # Skip files matching regex
    skip_folder_re:str=None, # Skip folders matching regex,
    maxdepth:int=None, # max depth to descend (1=just immediate contents; None=unlimited)
    symlinks:bool=True, # follow symlinks?
    ret_folders:bool=False, # return folders, not just files
    sort:bool=True, # sort files by name within each folder
    exts:str|list=None, # list or comma-separated str of exts to include
)->list: # Paths to matched files
    "Find files (optionally, also directories) based on file names"
    return list(globtastic(path, maxdepth=maxdepth, symlinks=symlinks, file_glob=file_glob, file_re=file_re,
        folder_re=folder_re, skip_file_glob=skip_file_glob, skip_file_re=skip_file_re, skip_folder_re=skip_folder_re,
        ret_folders=ret_folders, sort=sort, exts=exts))

In [ ]:
find_files('.', '*.css')

['./styles.css']

In [ ]:
#| export
def _load_gitignore(root='.'):
    "Compile .gitignore globs (plus .git) into one regex, or None"
    p = Path(root).expanduser()/'.gitignore'
    pats = [r'\.git/'] + ([l.strip() for l in p.read_text().splitlines() if l.strip() and not l.startswith(('#','!'))] if p.exists() else [])
    return re.compile('|'.join(translate(o) for o in pats)) if pats else None

def _grep_file(path, pat, s, after, before, context, nums):
    "Search one file; return formatted hit lines with context"
    if context: after=before=context
    lines = Path(path).expanduser().read_text(errors='ignore').splitlines()
    test = (lambda l: pat.search(l)) if pat else (lambda l: s in l)
    hits = [i for i,l in enumerate(lines) if test(l)]
    if not hits: return []
    ranges = []
    for i in hits:
        lo,hi = max(0,i-before),min(len(lines)-1,i+after)
        if ranges and lo<=ranges[-1][1]+1: ranges[-1][1] = max(ranges[-1][1],hi)
        else: ranges.append([lo,hi])
    hitset = set(hits)
    out = []
    for lo,hi in ranges:
        for i in range(lo,hi+1):
            sep = ':' if i in hitset else '-'
            out.append(f'{path}{sep}{i+1}{sep}{lines[i]}' if nums else f'{path}{sep}{lines[i]}')
    return out

In [ ]:
#| export
@delegates(find_files)
def grep_files(
    content_re:str=None, # Regex to search for in file contents
    content_str:str=None, # Plain string to search for in file contents
    path:str='.', # Path to start searching
    after:int=0, # Lines of context to show after each match
    before:int=0, # Lines of context to show before each match
    context:int=0, # Lines of context to show before and after each match
    nums:bool=True, # Include line numbers in output
    **kwargs
):
    "Search file contents like ripgrep, wrapping `find_files`"
    pat = re.compile(content_re) if content_re else None
    fs = find_files(path, **kwargs)
    return [l for f in fs for l in _grep_file(f, pat, content_str, after, before, context, nums)]

In [ ]:
grep_files('ce', file_glob='*.css', after=1)

['./styles.css:1:.cell {',
 './styles.css-2-  margin-bottom: 1rem;',
 './styles.css:5:.cell > .sourceCode {',
 './styles.css-6-  margin-bottom: 0;',
 './styles.css:9:.cell-output > pre {',
 './styles.css-10-  margin-bottom: 0;',
 './styles.css:13:.cell-output > pre, .cell-output > .sourceCode > pre, .cell-output-stdout > pre {',
 './styles.css-14-  margin-left: 0.8rem;',
 './styles.css:22:.cell-output > .sourceCode {',
 './styles.css-23-  border: none;',
 './styles.css:26:.cell-output > .sourceCode {',
 './styles.css-27-  background: none;']

In [ ]:
#| export
from pyskills.core import PosAllowPolicy,AllowPolicy,allow

In [ ]:
#| export
_wp = PosAllowPolicy(0)
allow(file_create, allow_policy=_wp);

## export -

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()